In [ ]:
# CELL 1 — Verify all libraries are available
import sys
print(f"Python version: {sys.version}")

libraries = ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "joblib"]
for lib in libraries:
    try:
        __import__(lib)
        print(f"  ✅ {lib} is installed")
    except ImportError:
        print(f"  ❌ {lib} is MISSING — run: pip install {lib}")


In [ ]:
# CELL 2 — Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
import joblib

print("✅ All libraries imported successfully!")


In [ ]:
# CELL 3 — Load the LIGTAS voltage dataset
df = pd.read_csv("ligtas_voltage_dataset.csv")

print("=== DATASET LOADED ===")
print(f"Total rows    : {len(df)}")
print(f"Total columns : {len(df.columns)}")
print(f"Columns       : {list(df.columns)}")
print("\nFirst 10 rows:")
display(df.head(10))
print("\nLast 5 rows:")
display(df.tail(5))


In [ ]:
# CELL 4 — Apply coverage area formula + all gap fixes
#
# FORMULA: r = 100 / (1 + voltage)   A = π × r²
#
# PHYSICS:
#   High voltage = source is NEAR  = small but intense danger zone
#   Low voltage  = source is FAR   = large but less intense danger zone
#   Weak source nearby → voltage still reads, gaps fixed below
#
# GAP FIXES:
#   GAP 1 — Weak source nearby      → voltage_class + danger_score
#   GAP 2 — Boundary zone 25-30V   → WARNING flag
#   GAP 3 — Class imbalance        → Safe zone oversampled
#   GAP 4 — 3 labels not 2         → Safe=0, Check=1, Dangerous=2

MAX_RADIUS = 100.0

# Coverage area (inverse: near strong source = small zone)
df["radius_m"]         = MAX_RADIUS / (1.0 + df["voltage_v"])
df["coverage_area_m2"] = (np.pi * df["radius_m"] ** 2).round(2)

# Voltage class 0–4
def get_voltage_class(v):
    if   v < 5.0:   return 0   # No leakage
    elif v < 15.0:  return 1   # Weak leakage
    elif v < 30.0:  return 2   # Moderate
    elif v < 100.0: return 3   # Dangerous
    else:           return 4   # Severe

df["voltage_class"] = df["voltage_v"].apply(get_voltage_class)

# Danger score — combined risk number
df["danger_score"] = (df["voltage_v"] / (1.0 + np.log1p(df["coverage_area_m2"]))).round(4)

# Risk flag
def get_risk_flag(row):
    v = row["voltage_v"]
    if   v >= 30.0: return "DANGER"
    elif v >= 25.0: return "WARNING"
    elif v >= 5.0:  return "CHECK"
    else:           return "SAFE"

df["risk_flag"] = df.apply(get_risk_flag, axis=1)

# 3-class label and status
def get_label(row):
    v = row["voltage_v"]
    if   v >= 30.0: return 2
    elif v >= 5.0:  return 1
    else:           return 0

def get_status(row):
    v = row["voltage_v"]
    if   v >= 30.0: return "Dangerous"
    elif v >= 5.0:  return "Check"
    else:           return "Safe"

df["label"]  = df.apply(get_label,  axis=1)
df["status"] = df.apply(get_status, axis=1)

# Oversample Safe zone (GAP 3 fix)
safe_extra = []
for v in np.round(np.arange(0.0, 5.0, 0.01), 2):
    if v in df["voltage_v"].values:
        continue
    r    = MAX_RADIUS / (1.0 + v)
    area = round(np.pi * r ** 2, 2)
    ds   = round(v / (1.0 + np.log1p(area)), 4)
    safe_extra.append({
        "voltage_v": v, "radius_m": round(r,4),
        "coverage_area_m2": area, "voltage_class": 0,
        "danger_score": ds, "risk_flag": "SAFE",
        "label": 0, "status": "Safe"
    })

df = pd.concat([df, pd.DataFrame(safe_extra)], ignore_index=True)
df = df.sort_values("voltage_v").reset_index(drop=True)
df = df[["voltage_v","coverage_area_m2","radius_m","voltage_class","danger_score","risk_flag","label","status"]]

print("✅ All features calculated!")
print("\nFormula: r = 100 / (1 + voltage)   →   A = π × r²")
print("\nSample results:")
for v in [0, 1, 5, 10, 15, 25, 29.9, 30.0, 50, 100, 220]:
    row = df[df["voltage_v"] == v]
    if not row.empty:
        r = row.iloc[0]
        print(f"  V={v:>6.1f}V  A={r['coverage_area_m2']:>10,.2f}m²  "
              f"class={r['voltage_class']}  score={r['danger_score']:>7.4f}  "
              f"flag={r['risk_flag']:<8}  [{r['status']}]")


In [ ]:
# CELL 5 — Data exploration
print("=== DATASET SUMMARY ===")
display(df.describe())

print("\n=== CLASS DISTRIBUTION (3 classes) ===")
counts = df["status"].value_counts()
print(counts)
print(f"\n  Safe      : {counts.get('Safe',      0):,} rows  (0.0V  –  4.9V)")
print(f"  Check     : {counts.get('Check',     0):,} rows  (5.0V  – 29.9V)")
print(f"  Dangerous : {counts.get('Dangerous', 0):,} rows  (30.0V – 1000V)")

print("\n=== RISK FLAG DISTRIBUTION ===")
print(df["risk_flag"].value_counts())


In [ ]:
# CELL 6 — Data visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("LIGTAS Dataset Overview", fontsize=14, fontweight="bold")

axes[0].hist(df["voltage_v"], bins=50, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
axes[0].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[0].set_title("Voltage Distribution")
axes[0].set_xlabel("Voltage (V)")
axes[0].set_ylabel("Count")
axes[0].legend()

colors = ["#2ecc71", "#f39c12", "#e74c3c"]
counts = df["status"].value_counts().reindex(["Safe", "Check", "Dangerous"])
axes[1].bar(counts.index, counts.values, color=colors, edgecolor="black")
axes[1].set_title("Class Distribution (3 Classes)")
axes[1].set_xlabel("Status")
axes[1].set_ylabel("Number of Samples")
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 50, str(v), ha="center", fontweight="bold")

sample = df[df["voltage_v"] <= 100].copy()
axes[2].plot(sample["voltage_v"], sample["coverage_area_m2"], color="steelblue", linewidth=2)
axes[2].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V Check")
axes[2].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                     where=(sample["voltage_v"] < 5),   color="green",  alpha=0.2, label="Safe")
axes[2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                     where=((sample["voltage_v"] >= 5) & (sample["voltage_v"] < 30)),
                     color="orange", alpha=0.2, label="Check")
axes[2].fill_between(sample["voltage_v"], sample["coverage_area_m2"],
                     where=(sample["voltage_v"] >= 30), color="red",    alpha=0.2, label="Dangerous")
axes[2].set_title("A = π × r²  Coverage Curve")
axes[2].set_xlabel("Voltage / Radius (V)")
axes[2].set_ylabel("Coverage Area (m²)")
axes[2].legend()
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("ligtas_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Chart saved as ligtas_overview.png")


In [ ]:
# CELL 7 — Prepare features (X) and labels (y)
# 4 features: voltage_v, coverage_area_m2, voltage_class, danger_score
# 3 labels:   0=Safe, 1=Check, 2=Dangerous

X = df[["voltage_v", "coverage_area_m2", "voltage_class", "danger_score"]]
y = df["label"]

print("=== FEATURES (X) ===")
display(X.head(10))

print("\n=== LABELS (y) ===")
print(f"  0 = Safe      : {(y == 0).sum():,} samples")
print(f"  1 = Check     : {(y == 1).sum():,} samples")
print(f"  2 = Dangerous : {(y == 2).sum():,} samples")
print(f"\nTotal : {len(y):,} samples")


In [ ]:
# CELL 8 — Train/Test split (80% train / 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== TRAIN / TEST SPLIT ===")
print(f"  Training samples : {len(X_train):,}  (80%)")
print(f"  Testing  samples : {len(X_test):,}  (20%)")
print("\nTraining label distribution:")
print(f"  Safe      : {(y_train == 0).sum():,}")
print(f"  Check     : {(y_train == 1).sum():,}")
print(f"  Dangerous : {(y_train == 2).sum():,}")
print("\nTesting label distribution:")
print(f"  Safe      : {(y_test == 0).sum():,}")
print(f"  Check     : {(y_test == 1).sum():,}")
print(f"  Dangerous : {(y_test == 2).sum():,}")


In [ ]:
# CELL 9 — Feature scaling using StandardScaler
# Prevents large coverage_area_m2 values from dominating voltage_v
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train ONLY, then transform
X_test_scaled  = scaler.transform(X_test)         # only transform — never fit on test

print("✅ Features scaled!")
print(f"\nBefore scaling (first row):")
print(f"  {X_train.iloc[0].to_dict()}")
print(f"\nAfter scaling (first row):")
for i, col in enumerate(X.columns):
    print(f"  {col:20s}: {X_train_scaled[0][i]:.6f}")


In [ ]:
# CELL 10 — Train Random Forest model
# Random Forest chosen over Logistic Regression because:
#   1. Handles non-linear relationships (voltage, area, danger score are non-linear)
#   2. Robust to class imbalance with class_weight="balanced"
#   3. No assumption about data distribution (unlike Logistic Regression)
#   4. Ensemble of 100 trees reduces overfitting risk
#   5. Proven superior for sensor-based IoT classification tasks
#      (Breiman, 2001; Boulesteix et al., 2018; water quality studies 2024)

print("Training Random Forest (100 trees)...")

rf_model = RandomForestClassifier(
    n_estimators=100,        # 100 decision trees vote together
    random_state=42,         # reproducible results
    class_weight="balanced"  # compensates for Safe/Check/Dangerous imbalance
)
rf_model.fit(X_train_scaled, y_train)

print("✅ Random Forest training complete!")
print(f"   Trees      : {rf_model.n_estimators}")
print(f"   Features   : {rf_model.n_features_in_}")
print(f"   Classes    : {list(rf_model.classes_)}  (0=Safe, 1=Check, 2=Dangerous)")


In [ ]:
# CELL 11 — Evaluate Random Forest model
rf_pred = rf_model.predict(X_test_scaled)

rf_acc  = accuracy_score(y_test,  rf_pred)
rf_prec = precision_score(y_test, rf_pred, average="weighted", zero_division=0)
rf_rec  = recall_score(y_test,    rf_pred, average="weighted", zero_division=0)
rf_f1   = f1_score(y_test,        rf_pred, average="weighted", zero_division=0)

print("=" * 50)
print("  MODEL: Random Forest")
print("=" * 50)
print(f"  Accuracy   : {rf_acc:.4f}  ({rf_acc*100:.2f}%)")
print(f"  Precision  : {rf_prec:.4f}")
print(f"  Recall     : {rf_rec:.4f}")
print(f"  F1-Score   : {rf_f1:.4f}")
print(f"\nDetailed Report:")
print(classification_report(y_test, rf_pred,
      target_names=["Safe (0)", "Check (1)", "Dangerous (2)"]))


In [ ]:
# CELL 12 — Model Metrics Chart
# Bar chart of Accuracy, Precision, Recall, F1-Score saved as model_metrics.png

metrics   = ["Accuracy", "Precision", "Recall", "F1-Score"]
rf_scores = [rf_acc, rf_prec, rf_rec, rf_f1]
colors    = ["#3b82f6", "#22c55e", "#f59e0b", "#ef4444"]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#161b22")

bars = ax.bar(metrics, rf_scores, color=colors, edgecolor="white",
              linewidth=1.2, width=0.5, zorder=3)

# Value labels on top of each bar
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f"{h:.4f}", ha="center", va="bottom",
            fontsize=11, color="white", fontweight="bold")

ax.set_ylim(0, 1.15)
ax.set_xticklabels(metrics, color="#e5e7eb", fontsize=12, fontweight="bold")
ax.set_ylabel("Score (0.0 – 1.0)", color="#9ca3af", fontsize=11)
ax.set_title("LIGTAS — Random Forest Model Performance Metrics",
             color="#f9fafb", fontsize=13, fontweight="bold", pad=14)
ax.tick_params(axis="both", colors="#6b7280", labelsize=11)
for spine in ax.spines.values(): spine.set_color("#30363d")
ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.0","0.2","0.4","0.6","0.8","1.0"], color="#6b7280")
ax.axhline(y=1.0, color="#374151", linestyle=":", linewidth=1.2, zorder=2)
ax.grid(axis="y", color="#21262d", linewidth=0.8, linestyle="--", zorder=1)

# Add a summary box
summary = f"Accuracy: {rf_acc:.4f}   Precision: {rf_prec:.4f}   Recall: {rf_rec:.4f}   F1: {rf_f1:.4f}"
fig.text(0.5, 0.01, summary, ha="center", color="#6b7280", fontsize=9, style="italic")

plt.tight_layout()
plt.savefig("model_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("✅ Metrics chart saved as model_metrics.png")


In [ ]:
# CELL 13 — Confusion Matrix visualization
fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle("LIGTAS — Random Forest Confusion Matrix", fontsize=13, fontweight="bold")

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Safe","Check","Dangerous"],
            yticklabels=["Safe","Check","Dangerous"],
            linewidths=0.5)
ax.set_xlabel("Predicted Label", fontsize=10)
ax.set_ylabel("Actual Label",    fontsize=10)

# TP/TN/FP/FN summary per class
print("Confusion Matrix Summary:")
for i, cls in enumerate(["Safe","Check","Dangerous"]):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    tn = cm.sum() - tp - fp - fn
    print(f"  {cls:<10} → TP={tp}  TN={tn}  FP={fp}  FN={fn}")

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrix saved as confusion_matrix.png")


In [ ]:
# CELL 14 — Coverage area curve visualization (3 zones)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("LIGTAS — Coverage Area Formula  A = π × r²", fontsize=13, fontweight="bold")

axes[0].plot(df["voltage_v"], df["coverage_area_m2"], color="steelblue", linewidth=1.5)
axes[0].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
axes[0].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[0].fill_between(df["voltage_v"], df["coverage_area_m2"],
                     where=(df["voltage_v"] < 5),   color="green",  alpha=0.3, label="Safe Zone")
axes[0].fill_between(df["voltage_v"], df["coverage_area_m2"],
                     where=((df["voltage_v"] >= 5) & (df["voltage_v"] < 30)),
                     color="orange", alpha=0.2, label="Check Zone")
axes[0].fill_between(df["voltage_v"], df["coverage_area_m2"],
                     where=(df["voltage_v"] >= 30), color="red",    alpha=0.2, label="Danger Zone")
axes[0].set_title("Full Range (0–1000V)")
axes[0].set_xlabel("Voltage / r (V)")
axes[0].set_ylabel("Coverage Area A (m²)")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.4)

zoom = df[df["voltage_v"] <= 100]
axes[1].plot(zoom["voltage_v"], zoom["coverage_area_m2"], color="steelblue", linewidth=2)
axes[1].axvline(x=5,  color="orange", linestyle="--", linewidth=1.5, label="5V  Check")
axes[1].axvline(x=30, color="red",    linestyle="--", linewidth=2,   label="30V Danger")
axes[1].fill_between(zoom["voltage_v"], zoom["coverage_area_m2"],
                     where=(zoom["voltage_v"] < 5),  color="green",  alpha=0.3, label="Safe")
axes[1].fill_between(zoom["voltage_v"], zoom["coverage_area_m2"],
                     where=((zoom["voltage_v"] >= 5) & (zoom["voltage_v"] < 30)),
                     color="orange", alpha=0.2, label="Check")
axes[1].fill_between(zoom["voltage_v"], zoom["coverage_area_m2"],
                     where=(zoom["voltage_v"] >= 30), color="red",   alpha=0.3, label="Dangerous")
axes[1].annotate("30V\n32.7 m²", xy=(30, np.pi*(100/(1+30))**2),
                 xytext=(50, 1000), arrowprops=dict(arrowstyle="->", color="black"),
                 fontsize=9, color="red")
axes[1].set_title("Zoomed (0–100V) — Threshold Detail")
axes[1].set_xlabel("Voltage / r (V)")
axes[1].set_ylabel("Coverage Area A (m²)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("coverage_area_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Coverage curve saved as coverage_area_curve.png")


In [ ]:
# CELL 15 — Save trained model and scaler
joblib.dump(rf_model, "ligtas_rf_model.pkl")
joblib.dump(scaler,   "ligtas_scaler.pkl")
print("✅ Files saved:")
print("   📄 ligtas_rf_model.pkl  — trained Random Forest model")
print("   📄 ligtas_scaler.pkl    — feature scaler (4 features)")
print("\nThese files can be loaded anytime without retraining.")


In [ ]:
# CELL 16 — Real-time prediction function (simulates ESP32 input)
def ligtas_predict(voltage_value):
    r    = MAX_RADIUS / (1.0 + voltage_value)
    area = np.pi * (r ** 2)
    if   voltage_value < 5.0:   vc = 0
    elif voltage_value < 15.0:  vc = 1
    elif voltage_value < 30.0:  vc = 2
    elif voltage_value < 100.0: vc = 3
    else:                       vc = 4
    ds = voltage_value / (1.0 + np.log1p(area))
    input_scaled = scaler.transform([[voltage_value, area, vc, ds]])
    pred = rf_model.predict(input_scaled)[0]
    if   pred == 0: status = "🟢 SAFE — No Electrical Leakage"
    elif pred == 1: status = "🟡 CHECK — Possible Weak Leakage"
    else:           status = "🔴 DANGEROUS — Electrical Leakage Detected!"
    print(f"  ┌───────────────────────────────────────────────")
    print(f"  │  Voltage Input    : {voltage_value} V")
    print(f"  │  Radius           : {r:.2f} m")
    print(f"  │  Coverage Area    : {area:,.2f} m²")
    print(f"  │  Voltage Class    : {vc}  (0=none 1=weak 2=mod 3=danger 4=severe)")
    print(f"  │  Danger Score     : {ds:.4f}")
    print(f"  │  Classification   : {status}")
    print(f"  └───────────────────────────────────────────────")

print("=" * 55)
print("  LIGTAS — Voltage Prediction Test")
print("=" * 55)
for v in [0.0, 1.0, 5.0, 10.0, 15.0, 25.0, 29.9, 30.0, 50.0, 110.0, 220.0, 1000.0]:
    ligtas_predict(v)


In [ ]:
# CELL 17 — Load saved model without retraining
loaded_model  = joblib.load("ligtas_rf_model.pkl")
loaded_scaler = joblib.load("ligtas_scaler.pkl")
print("✅ Model and scaler loaded from files!")

test_v  = 35.0
r       = MAX_RADIUS / (1.0 + test_v)
area    = np.pi * r ** 2
vc      = 3
ds      = test_v / (1.0 + np.log1p(area))
tinput  = loaded_scaler.transform([[test_v, area, vc, ds]])
result  = loaded_model.predict(tinput)[0]
labels  = {0: "🟢 SAFE", 1: "🟡 CHECK", 2: "🔴 DANGEROUS"}
print(f"\nQuick test — Voltage: {test_v}V")
print(f"  Coverage Area : {area:.2f} m²")
print(f"  Danger Score  : {ds:.4f}")
print(f"  Result        : {labels[result]}")


In [ ]:
# CELL 18 — Load existing model and export to Arduino C++
import joblib, numpy as np
from micromlgen import port

rf_model = joblib.load("ligtas_rf_model.pkl")
scaler   = joblib.load("ligtas_scaler.pkl")

with open("ligtas_model.h", "w") as f:
    f.write(port(rf_model))
print("✅ ligtas_model.h created!")

print("\n=== COPY THESE INTO ligtas_ml.h ===")
print(f"const float SCALER_MEAN[4]  = {{{scaler.mean_[0]:.6f}f, {scaler.mean_[1]:.6f}f, {scaler.mean_[2]:.6f}f, {scaler.mean_[3]:.6f}f}};")
print(f"const float SCALER_SCALE[4] = {{{scaler.scale_[0]:.6f}f, {scaler.scale_[1]:.6f}f, {scaler.scale_[2]:.6f}f, {scaler.scale_[3]:.6f}f}};")
